# Prompt 8: Built-in Functions and Expressions

**Exam Objective 4 — Terraform Configuration (HCL)**  
**HashiCorp Certified: Terraform Associate (004)**

---

## Topics Covered

| Section | Topic |
|---|---|
| 1 | `terraform console` — interactive function testing |
| 2 | String functions |
| 3 | Numeric functions |
| 4 | Collection functions ⭐ HIGH PRIORITY |
| 5 | Type conversion functions |
| 6 | Encoding functions |
| 7 | Filesystem functions |
| 8 | Date/time functions |
| 9 | IP networking functions |
| 10 | Conditional expression |
| 11 | `can()` and `try()` functions |
| 12 | Exam-style questions |
| 13 | Real-world scenarios |
| 14 | Quick-reference cheat sheet |

---
## 1. `terraform console` — Interactive Function Testing

`terraform console` opens a **REPL (Read-Eval-Print Loop)** that lets you test HCL expressions and functions interactively — without writing a configuration file or running `terraform apply`.

### Starting the Console

```bash
# Run from your working directory (providers must be initialized)
terraform console
```

### Console Capabilities

```
> upper("hello world")
"HELLO WORLD"

> length(["a", "b", "c"])
3

> cidrsubnet("10.0.0.0/16", 8, 3)
"10.0.3.0/24"

> jsonencode({name = "alice", age = 30})
{"age":30,"name":"alice"}

# Access variables from your configuration
> var.environment
"production"

# Access resource attributes from state
> aws_instance.web.public_ip
"54.123.45.67"

# Exit
> exit
```

### Key Points
- `terraform console` **reads state** — it can access resource attributes and variable values.
- It does **not** apply any changes — purely read-only for infrastructure.
- Essential for debugging complex expressions before embedding them in configuration.
- Requires `terraform init` to have been run (needs provider schemas).
- Exit with `Ctrl+C`, `Ctrl+D`, or typing `exit`.

---
## 2. String Functions

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│ FUNCTION          │ DESCRIPTION                          │ EXAMPLE / OUTPUT      │
├───────────────────┼──────────────────────────────────────┼───────────────────────┤
│ upper(str)        │ Uppercase all characters             │ upper("hello")        │
│                   │                                      │ → "HELLO"             │
│ lower(str)        │ Lowercase all characters             │ lower("HELLO")        │
│                   │                                      │ → "hello"             │
│ trimspace(str)    │ Strip leading and trailing whitespace│ trimspace(" hi ")     │
│                   │                                      │ → "hi"                │
│ split(sep, str)   │ Split string into list               │ split(",","a,b,c")   │
│                   │                                      │ → ["a","b","c"]       │
│ join(sep, list)   │ Join list into string                │ join("-",["a","b"])   │
│                   │                                      │ → "a-b"               │
│ replace(s,old,new)│ Replace all occurrences of old       │ replace("a-b","-","_")│
│                   │                                      │ → "a_b"               │
│ substr(str,off,ln)│ Extract substring (offset, length)   │ substr("hello",1,3)   │
│                   │                                      │ → "ell"               │
│ format(fmt, ...)  │ sprintf-style string formatting      │ format("%s-%d","v",2) │
│                   │                                      │ → "v-2"               │
│ formatlist(fmt,..)│ format() applied to each list element│ formatlist("i-%s",    │
│                   │                                      │ ["a","b"]) → ["i-a","i-b"]│
│ regex(pattern,str)│ Match pattern, return captures       │ regex("[a-z]+","abc1")│
│                   │                                      │ → "abc"               │
│ regexall(pat,str) │ All non-overlapping matches as list  │ regexall("[0-9]+",    │
│                   │                                      │ "1a2b3") → ["1","2","3"]│
│ startswith(s,pfx) │ True if str starts with prefix       │ startswith("us-east", │
│                   │                                      │ "us") → true          │
│ endswith(s,sfx)   │ True if str ends with suffix         │ endswith("prod.com",  │
│                   │                                      │ ".com") → true        │
└───────────────────┴──────────────────────────────────────┴───────────────────────┘
```

### String Function Code Examples

```hcl
locals {
  # Build a consistent resource name from parts
  app      = "MyApp"
  env      = "  Production  "
  region   = "us-east-1"

  clean_env    = lower(trimspace(local.env))                        # "production"
  resource_name = format("%s-%s-%s", lower(local.app), local.clean_env, local.region)
  # "myapp-production-us-east-1"

  # Parse a CSV variable value
  csv_tags     = "web,app,db,cache"
  tag_list     = split(",", local.csv_tags)   # ["web", "app", "db", "cache"]
  tag_string   = join(" | ", local.tag_list)  # "web | app | db | cache"

  # Sanitize a bucket name (replace dots with dashes)
  raw_name     = "my.company.assets"
  bucket_name  = replace(local.raw_name, ".", "-")   # "my-company-assets"

  # Generate a list of formatted names
  zones        = ["a", "b", "c"]
  zone_ids     = formatlist("us-east-1%s", local.zones)
  # ["us-east-1a", "us-east-1b", "us-east-1c"]
}
```

---
## 3. Numeric Functions

```hcl
# abs() — absolute value
abs(-5)       # 5
abs(3.7)      # 3.7

# ceil() — round up to nearest integer
ceil(1.1)     # 2
ceil(4.0)     # 4

# floor() — round down to nearest integer
floor(4.9)    # 4
floor(-1.1)   # -2

# max() — largest of all arguments
max(3, 1, 7, 2)    # 7
max(5)             # 5

# min() — smallest of all arguments
min(3, 1, 7, 2)    # 1

# parseint() — parse string to integer in given base
parseint("FF", 16)   # 255 (hex to decimal)
parseint("1010", 2)  # 10  (binary to decimal)
parseint("42", 10)   # 42

# signum() — sign of a number: -1, 0, or 1
signum(-5)    # -1
signum(0)     #  0
signum(100)   #  1
```

### Practical Use

```hcl
variable "desired_replicas" {
  type    = number
  default = 2.7
}

locals {
  # Always round up for minimum capacity guarantees
  safe_replicas = ceil(var.desired_replicas)   # 3

  # Clamp to a minimum of 2
  min_replicas  = max(2, ceil(var.desired_replicas))   # max(2, 3) = 3
}
```

---
## 4. Collection Functions ⭐ HIGH EXAM PRIORITY

### `length()` — Element Count

```hcl
length(["a", "b", "c"])           # 3  — list
length({a = 1, b = 2})            # 2  — map
length("hello")                   # 5  — string (character count)
```

### `contains()` — Membership Test

```hcl
contains(["web", "app", "db"], "app")   # true
contains(["web", "app", "db"], "cache") # false
```

### `keys()` and `values()` — Map Decomposition

```hcl
variable "tags" {
  default = { Environment = "prod", Team = "platform" }
}

keys(var.tags)    # ["Environment", "Team"]  — sorted alphabetically
values(var.tags)  # ["prod", "platform"]     — in key-sorted order
```

### `lookup()` ⭐ — Safe Map Value Retrieval

```hcl
variable "ami_ids" {
  type = map(string)
  default = {
    "us-east-1" = "ami-0abc123"
    "us-west-2" = "ami-0def456"
  }
}

# lookup(map, key, default) — returns default if key not found
lookup(var.ami_ids, "us-east-1", "ami-unknown")   # "ami-0abc123"
lookup(var.ami_ids, "eu-west-1", "ami-unknown")   # "ami-unknown" (key missing)

# Direct access ERRORS if key is missing:
# var.ami_ids["eu-west-1"]  # Error: key not found
```

### `merge()` ⭐ — Combine Maps

```hcl
locals {
  common_tags = {
    Owner      = "platform-team"
    ManagedBy  = "terraform"
  }
  resource_tags = {
    Environment = "production"
    Name        = "web-server"
  }

  # merge() — later maps override earlier ones for duplicate keys
  all_tags = merge(local.common_tags, local.resource_tags)
  # Result:
  # {
  #   Owner       = "platform-team"
  #   ManagedBy   = "terraform"
  #   Environment = "production"
  #   Name        = "web-server"
  # }

  # Override a common tag for a specific resource
  db_tags = merge(local.common_tags, {
    Name  = "primary-db"
    Owner = "db-team"   # overrides common_tags.Owner
  })
}
```

### `flatten()` ⭐ — Flatten Nested Lists

```hcl
# Flattens one level deep by default, but fully flattens nested lists
flatten([["a", "b"], ["c", "d"]])          # ["a", "b", "c", "d"]
flatten([["a", ["b", "c"]], ["d"]])        # ["a", "b", "c", "d"]

# Practical: collect all subnet IDs across multiple VPCs
locals {
  vpc_subnets = [
    ["subnet-aaa", "subnet-bbb"],   # vpc-1 subnets
    ["subnet-ccc"],                  # vpc-2 subnets
    ["subnet-ddd", "subnet-eee"],   # vpc-3 subnets
  ]
  all_subnets = flatten(local.vpc_subnets)
  # ["subnet-aaa", "subnet-bbb", "subnet-ccc", "subnet-ddd", "subnet-eee"]
}
```

### `distinct()` — Remove Duplicates

```hcl
distinct(["web", "app", "web", "db", "app"])   # ["web", "app", "db"]
```

### `concat()` — Combine Lists

```hcl
concat(["a", "b"], ["c", "d"], ["e"])   # ["a", "b", "c", "d", "e"]
```

### `element()` — Access List Element by Index (with Wrap-around)

```hcl
element(["a", "b", "c"], 0)   # "a"
element(["a", "b", "c"], 4)   # "b"  — wraps: 4 % 3 = 1
# Useful for distributing resources across AZs:
element(var.availability_zones, count.index)
```

### `index()` — Find Position of Value in List

```hcl
index(["a", "b", "c"], "b")   # 1
index(["a", "b", "c"], "z")   # Error — value not found
```

### `range()` — Generate Numeric List

```hcl
range(5)          # [0, 1, 2, 3, 4]
range(1, 5)       # [1, 2, 3, 4]
range(0, 10, 2)   # [0, 2, 4, 6, 8]  — start, end, step
```

### `zipmap()` — Build Map from Parallel Lists

```hcl
# zipmap(keys_list, values_list)
zipmap(["a", "b", "c"], [1, 2, 3])
# {"a" = 1, "b" = 2, "c" = 3}

# Practical: build an instance-ID-to-name map
locals {
  names = ["web", "app", "db"]
  ids   = ["i-001", "i-002", "i-003"]
  id_map = zipmap(local.ids, local.names)
  # {"i-001" = "web", "i-002" = "app", "i-003" = "db"}
}
```

### Collection Function Summary

```
┌────────────────┬──────────────────────────────────────────────────────────┐
│ Function       │ Key Behaviour                                            │
├────────────────┼──────────────────────────────────────────────────────────┤
│ length()       │ Works on lists, maps, sets, strings                      │
│ contains()     │ Checks if value is in a list or set                      │
│ keys()         │ Returns sorted list of map keys                          │
│ values()       │ Returns list of values in key-sorted order               │
│ lookup() ⭐    │ Safe map access with default — avoids errors             │
│ merge()  ⭐    │ Merges N maps; later keys override earlier               │
│ flatten() ⭐   │ Flattens nested lists into a single flat list            │
│ distinct()     │ Removes duplicates, preserves first occurrence           │
│ concat()       │ Combines multiple lists into one                         │
│ element()      │ Index access with modulo wrap-around                     │
│ index()        │ Returns position of value; errors if not found           │
│ range()        │ Generates integer sequence                               │
│ zipmap()       │ Builds map from two parallel lists                       │
└────────────────┴──────────────────────────────────────────────────────────┘
```

---
## 5. Type Conversion Functions

```hcl
# tostring() — convert any scalar to string
tostring(42)       # "42"
tostring(3.14)     # "3.14"
tostring(true)     # "true"
tostring(false)    # "false"
tostring(null)     # null  (null stays null)

# tonumber() — convert string or bool to number
tonumber("42")     # 42
tonumber("3.14")   # 3.14
tonumber(true)     # 1
tonumber(false)    # 0
tonumber("abc")    # Error — not numeric

# tobool() — convert string to bool
tobool("true")     # true
tobool("false")    # false
tobool("yes")      # Error — only "true" / "false" accepted
tobool(1)          # Error — only string input
```

### Using Type Conversion in Variable Validation

```hcl
variable "port" {
  type        = string
  description = "Port number as string"

  validation {
    condition     = can(tonumber(var.port)) && tonumber(var.port) >= 1 && tonumber(var.port) <= 65535
    error_message = "Port must be a number between 1 and 65535."
  }
}
```

---
## 6. Encoding Functions

### `jsonencode()` and `jsondecode()` ⭐

```hcl
# jsonencode() — convert HCL value to JSON string
locals {
  policy = {
    Version = "2012-10-17"
    Statement = [{
      Effect   = "Allow"
      Action   = ["s3:GetObject", "s3:ListBucket"]
      Resource = "arn:aws:s3:::my-bucket/*"
    }]
  }

  policy_json = jsonencode(local.policy)
  # {"Statement":[{"Action":["s3:GetObject","s3:ListBucket"],"Effect":"Allow","Resource":"..."}],"Version":"2012-10-17"}
}

resource "aws_iam_role_policy" "example" {
  role   = aws_iam_role.example.id
  policy = jsonencode({
    Version = "2012-10-17"
    Statement = [{
      Effect   = "Allow"
      Action   = "*"
      Resource = "*"
    }]
  })
}

# jsondecode() — parse a JSON string to HCL value
locals {
  raw_json = file("config.json")         # read file contents
  config   = jsondecode(local.raw_json)  # parse to HCL object
  region   = local.config.region        # access a field
}
```

### `yamlencode()` and `yamldecode()`

```hcl
# yamlencode() — convert HCL to YAML string
yamlencode({
  name    = "alice"
  enabled = true
  ports   = [80, 443]
})
# ---
# enabled: true
# name: alice
# ports:
# - 80
# - 443

# yamldecode() — parse YAML string to HCL value
locals {
  raw_yaml = file("values.yaml")
  parsed   = yamldecode(local.raw_yaml)
}
```

### `base64encode()` and `base64decode()`

```hcl
base64encode("hello world")          # "aGVsbG8gd29ybGQ="
base64decode("aGVsbG8gd29ybGQ=")     # "hello world"

# Common use: EC2 user_data
resource "aws_instance" "web" {
  ami           = var.ami_id
  instance_type = "t3.micro"
  user_data     = base64encode(file("user_data.sh"))
}
```

---
## 7. Filesystem Functions

### `file()` — Read File Contents as String

```hcl
# Read a shell script for EC2 user_data
resource "aws_instance" "web" {
  user_data = file("${path.module}/scripts/bootstrap.sh")
}

# Read an SSH public key
resource "aws_key_pair" "deployer" {
  key_name   = "deployer"
  public_key = file("~/.ssh/id_rsa.pub")
}
```

**Important:** `file()` runs at **plan time**, so the file must exist on the machine running Terraform.

### `filebase64()` — Read File as Base64

```hcl
# Some resources require Base64-encoded content
resource "aws_lambda_function" "example" {
  filename      = "lambda_payload.zip"
  function_name = "my-function"
  # filebase64() is not needed here — filename is used directly
  # but useful in other contexts:
}

# Container registry image base64
locals {
  cert_b64 = filebase64("tls/cert.pem")
}
```

### `templatefile()` ⭐ — Read File and Render Template Variables

`templatefile(path, vars)` reads a file and substitutes `${variable}` expressions using the provided map.

```
# File: templates/user_data.sh.tftpl
#!/bin/bash
hostnamectl set-hostname ${hostname}
apt-get install -y ${package}
echo "Environment: ${environment}" >> /etc/motd
```

```hcl
resource "aws_instance" "web" {
  ami           = var.ami_id
  instance_type = "t3.micro"

  user_data = templatefile("${path.module}/templates/user_data.sh.tftpl", {
    hostname    = "web-${terraform.workspace}"
    package     = "nginx"
    environment = var.environment
  })
}
```

**Rendered result:** `${hostname}` becomes the actual hostname value at plan time.

**File extension:** Terraform convention uses `.tftpl` for template files to distinguish from `.sh` or `.json`.

### Path References Used with Filesystem Functions

```
path.module   — directory of the current module's .tf files
path.root     — directory of the root module
path.cwd      — current working directory when terraform was invoked
```

---
## 8. Date/Time Functions

```hcl
# timestamp() — RFC 3339 UTC timestamp at plan time
timestamp()   # e.g., "2026-04-23T20:00:00Z"

# Caution: timestamp() changes on every plan → causes perpetual drift
# Use ignore_changes to suppress this:
resource "aws_instance" "web" {
  tags = {
    CreatedAt = timestamp()
  }
  lifecycle {
    ignore_changes = [tags["CreatedAt"]]
  }
}

# formatdate() — format a timestamp string
# formatdate(format, timestamp)
formatdate("YYYY-MM-DD", timestamp())   # e.g., "2026-04-23"
formatdate("DD MMM YYYY hh:mm UTC", "2026-04-23T15:30:00Z")
# "23 Apr 2026 15:30 UTC"

# timeadd() — add a duration to a timestamp
# timeadd(timestamp, duration)
timeadd("2026-04-23T00:00:00Z", "24h")    # "2026-04-24T00:00:00Z"
timeadd("2026-04-23T00:00:00Z", "1h30m")  # "2026-04-23T01:30:00Z"
timeadd("2026-04-23T00:00:00Z", "-72h")   # 3 days in the past
```

**Exam tip:** `timestamp()` is evaluated at plan time and changes every run — always use `ignore_changes` when storing it as a resource tag.

---
## 9. IP Networking Functions ⭐

These functions are highly practical for VPC/subnet calculations and appear frequently in networking-focused configurations.

### `cidrsubnet()` ⭐ — Calculate a Subnet CIDR

```
cidrsubnet(prefix, newbits, netnum)
  prefix  — base CIDR block (e.g., "10.0.0.0/16")
  newbits — bits to add to the prefix length
  netnum  — subnet number (zero-based)
```

```hcl
# Base: 10.0.0.0/16 (65536 addresses)
# Adding 8 bits gives /24 subnets (256 each)

cidrsubnet("10.0.0.0/16", 8, 0)   # "10.0.0.0/24"
cidrsubnet("10.0.0.0/16", 8, 1)   # "10.0.1.0/24"
cidrsubnet("10.0.0.0/16", 8, 3)   # "10.0.3.0/24"

# Adding 4 bits gives /20 subnets
cidrsubnet("10.0.0.0/16", 4, 0)   # "10.0.0.0/20"
cidrsubnet("10.0.0.0/16", 4, 1)   # "10.0.16.0/20"

# Practical: auto-generate subnets for count-based resources
resource "aws_subnet" "private" {
  count             = 3
  vpc_id            = aws_vpc.main.id
  cidr_block        = cidrsubnet(aws_vpc.main.cidr_block, 8, count.index)
  availability_zone = element(var.availability_zones, count.index)
}
```

### `cidrhost()` — Calculate a Host IP Within a CIDR

```hcl
# cidrhost(prefix, hostnum) — host number within the CIDR
cidrhost("10.0.0.0/24", 1)     # "10.0.0.1"   — first host
cidrhost("10.0.0.0/24", 254)   # "10.0.0.254" — last usable host
cidrhost("10.0.0.0/24", -2)    # "10.0.0.253" — negative counts from end
```

### `cidrnetmask()` — Subnet Mask from CIDR

```hcl
cidrnetmask("10.0.0.0/24")   # "255.255.255.0"
cidrnetmask("10.0.0.0/16")   # "255.255.0.0"
cidrnetmask("10.0.0.0/8")    # "255.0.0.0"
```

### Automated VPC Subnet Layout Example

```hcl
variable "vpc_cidr" {
  default = "10.0.0.0/16"
}

variable "availability_zones" {
  default = ["us-east-1a", "us-east-1b", "us-east-1c"]
}

resource "aws_vpc" "main" {
  cidr_block = var.vpc_cidr
}

resource "aws_subnet" "public" {
  count             = length(var.availability_zones)
  vpc_id            = aws_vpc.main.id
  # Public: 10.0.0.0/24, 10.0.1.0/24, 10.0.2.0/24
  cidr_block        = cidrsubnet(var.vpc_cidr, 8, count.index)
  availability_zone = var.availability_zones[count.index]
  map_public_ip_on_launch = true
}

resource "aws_subnet" "private" {
  count             = length(var.availability_zones)
  vpc_id            = aws_vpc.main.id
  # Private: 10.0.10.0/24, 10.0.11.0/24, 10.0.12.0/24
  cidr_block        = cidrsubnet(var.vpc_cidr, 8, count.index + 10)
  availability_zone = var.availability_zones[count.index]
}
```

---
## 10. Conditional Expression

The conditional expression is Terraform's inline `if/else` — it evaluates to one of two values based on a boolean condition.

### Syntax

```
condition ? true_value : false_value
```

### Examples

```hcl
variable "environment" {
  type    = string
  default = "production"
}

locals {
  # Choose instance type based on environment
  instance_type = var.environment == "production" ? "t3.large" : "t3.micro"

  # Enable deletion protection in prod
  deletion_protection = var.environment == "production" ? true : false

  # Ternary with function calls
  bucket_name = var.environment == "production" ? "prod-assets" : lower("${var.environment}-assets")
}

resource "aws_instance" "web" {
  count         = var.environment == "production" ? 3 : 1
  ami           = var.ami_id
  instance_type = local.instance_type
}
```

### Null Coalescing Pattern

```hcl
# If a variable is null, use a default value
variable "custom_name" {
  type    = string
  default = null
}

locals {
  effective_name = var.custom_name != null ? var.custom_name : "default-name"
}
```

### Key Rules
- Both branches **must produce the same type** (Terraform enforces type consistency).
- Conditionals can be nested, but deeply nested ternaries reduce readability — use `locals` to break them up.
- The condition must evaluate to a `bool` — Terraform will coerce `0`/`1` and `"true"`/`"false"` strings if needed.

---
## 11. `can()` and `try()` Functions

### `can()` — Test If an Expression Evaluates Without Error

`can(expression)` returns `true` if the expression evaluates successfully, `false` if it would produce an error.

```hcl
can(tonumber("42"))    # true  — "42" is valid numeric string
can(tonumber("abc"))   # false — "abc" cannot convert to number
can(var.config.port)   # true if var.config has a .port attribute

# Primary use: variable validation conditions
variable "instance_type" {
  type = string

  validation {
    condition = can(regex("^t[23]\\.", var.instance_type))
    error_message = "Instance type must be a t2 or t3 family type."
  }
}

# Validate a port number provided as string
variable "port" {
  type = string

  validation {
    condition = can(tonumber(var.port)) && tonumber(var.port) > 0
    error_message = "Port must be a positive numeric string."
  }
}
```

### `try()` — Return First Expression That Succeeds

`try(expr1, expr2, ..., fallback)` evaluates each expression in order and returns the **first one that succeeds without error**. The last argument acts as the default if all others fail.

```hcl
# Safely access a potentially absent attribute
variable "config" {
  type = any
  default = { environment = "production" }
  # Note: no 'port' key
}

locals {
  port = try(var.config.port, 8080)         # 8080 — var.config.port doesn't exist
  env  = try(var.config.environment, "dev") # "production" — key exists
}

# Parse a value that might be a number or a string
locals {
  raw_value = "42"
  as_number = try(tonumber(local.raw_value), 0)   # 42

  raw_value2 = "not-a-number"
  as_number2 = try(tonumber(local.raw_value2), 0) # 0 — fallback
}
```

### `can()` vs `try()` — Decision Guide

```
┌────────────┬──────────────────────────────────────────────────────────────┐
│ Function   │ Use When                                                     │
├────────────┼──────────────────────────────────────────────────────────────┤
│ can()      │ You need a boolean result (e.g., in a validation condition)  │
│ try()      │ You want to use the value directly, with a fallback          │
│            │ You're accessing optional attributes of a complex object     │
└────────────┴──────────────────────────────────────────────────────────────┘
```

---
## 12. Exam-Style Practice Questions

---

**Q1: A Terraform configuration needs to safely retrieve a value from a map variable, returning a default string `"unknown"` if the key does not exist. Which function achieves this?**

A) `contains(var.my_map, "my_key")`  
B) `index(var.my_map, "my_key")`  
C) `lookup(var.my_map, "my_key", "unknown")`  
D) `element(var.my_map, "my_key")`  

<details>
<summary>Answer</summary>

**Answer: C**  
`lookup(map, key, default)` is the correct function for safe map access with a fallback. It returns the value for the key if found, or the default value if the key is absent — without raising an error. Option A (`contains`) tests membership but returns a bool, not the value. Option B (`index`) is for lists, not maps, and errors if the value isn't found. Option D (`element`) is for lists accessed by numeric index, not maps.

</details>

---

**Q2: An engineer needs to merge a `common_tags` map with a `resource_tags` map. A key `Name` exists in both maps. Which statement about `merge(common_tags, resource_tags)` is correct?**

A) `merge()` raises an error when duplicate keys are present  
B) The `Name` value from `common_tags` (the first argument) takes precedence  
C) The `Name` value from `resource_tags` (the last argument) takes precedence  
D) `merge()` produces a list of maps when keys conflict  

<details>
<summary>Answer</summary>

**Answer: C**  
When `merge()` encounters duplicate keys, **the value from the rightmost (last) map argument wins**. This means `resource_tags.Name` overrides `common_tags.Name`. This behaviour is intentional — it allows a pattern where you pass a common base map as the first argument and specific overrides as later arguments. Option A is false — no error is raised. Option B has the precedence backwards. Option D is false — the output is always a single merged map.

</details>

---

**Q3: A Terraform engineer wants to calculate the CIDR block for the 4th private subnet (index 3) of a VPC with CIDR `172.16.0.0/16`, adding 8 bits to create `/24` subnets. Which expression is correct?**

A) `cidrsubnet("172.16.0.0/16", 3, 8)`  
B) `cidrsubnet("172.16.0.0/16", 8, 3)`  
C) `cidrhost("172.16.0.0/16", 8)`  
D) `cidrnetmask("172.16.0.0/16", 8, 3)`  

<details>
<summary>Answer</summary>

**Answer: B**  
`cidrsubnet(prefix, newbits, netnum)` — the argument order is: base CIDR, bits to add, subnet number. `cidrsubnet("172.16.0.0/16", 8, 3)` produces `"172.16.3.0/24"` — a /24 subnet (16+8=24 prefix bits), the 4th subnet (0-indexed as 3). Option A swaps `newbits` and `netnum` — `cidrsubnet("172.16.0.0/16", 3, 8)` would give a /19 subnet numbered 8, which is `"172.16.64.0/19"`. Option C (`cidrhost`) calculates a host address within a CIDR, not a subnet. Option D is not valid — `cidrnetmask` only accepts one argument (the CIDR prefix).

</details>

---
## 13. Real-World Scenarios

<details>
<summary>Scenario 1: Centralised Tagging Strategy Using merge() and locals</summary>

**Situation:**  
A platform team needs every AWS resource tagged with common organisation-wide tags plus resource-specific tags. They want to define common tags once and merge them into each resource.

```hcl
# variables.tf
variable "environment" {
  type    = string
  default = "production"
}

variable "cost_centre" {
  type    = string
  default = "engineering"
}

# locals.tf
locals {
  common_tags = {
    Environment = var.environment
    ManagedBy   = "terraform"
    CostCentre  = var.cost_centre
    Owner       = "platform-team"
  }
}

# main.tf
resource "aws_instance" "web" {
  ami           = var.ami_id
  instance_type = var.environment == "production" ? "t3.large" : "t3.micro"

  tags = merge(local.common_tags, {
    Name        = "web-server-01"
    Application = "frontend"
  })
}

resource "aws_db_instance" "primary" {
  # ... db config ...
  tags = merge(local.common_tags, {
    Name  = "primary-database"
    Owner = "db-team"   # overrides common_tags.Owner for this resource
  })
}
```

**Console verification:**
```bash
terraform console
> merge({A="1", B="2"}, {B="override", C="3"})
{"A" = "1", "B" = "override", "C" = "3"}
```

**Expected outcome:** All resources receive the four common tags. Resource-specific tags are added, and when a key conflicts (like `Owner`), the resource-specific value wins.  
**Exam sub-objective:** Objective 4 — `merge()` function; `locals` block; conditional expression.

</details>

<details>
<summary>Scenario 2: Dynamic EC2 User Data with templatefile()</summary>

**Situation:**  
An operations team needs to bootstrap EC2 instances with different configurations based on the server role (web, app, db). They use `templatefile()` to avoid duplicating similar shell scripts.

```bash
# templates/bootstrap.sh.tftpl
#!/bin/bash
set -e
hostnamectl set-hostname ${hostname}
echo "${environment}" > /etc/server-environment

%{ for pkg in packages ~}
apt-get install -y ${pkg}
%{ endfor ~}

%{ if enable_monitoring ~}
curl -fsSL https://metrics.example.com/install.sh | bash
%{ endif ~}
```

```hcl
variable "servers" {
  type = map(object({
    packages          = list(string)
    enable_monitoring = bool
  }))
  default = {
    web = { packages = ["nginx", "certbot"], enable_monitoring = true  }
    app = { packages = ["nodejs", "pm2"],   enable_monitoring = true  }
    db  = { packages = ["mysql-server"],     enable_monitoring = false }
  }
}

resource "aws_instance" "servers" {
  for_each      = var.servers
  ami           = var.ami_id
  instance_type = "t3.micro"

  user_data = templatefile("${path.module}/templates/bootstrap.sh.tftpl", {
    hostname          = "${each.key}-server"
    environment       = var.environment
    packages          = each.value.packages
    enable_monitoring = each.value.enable_monitoring
  })
}
```

**Expected outcome:** Each server type gets a customised bootstrap script with its specific packages and monitoring configuration, generated at plan time.  
**Exam sub-objective:** Objective 4 — `templatefile()` filesystem function; `for_each` with map.

</details>

<details>
<summary>Scenario 3: Automated Multi-Tier VPC Subnet Layout with cidrsubnet()</summary>

**Situation:**  
A network engineer needs to provision a VPC with three tiers (public, private, database) across three availability zones — 9 subnets total — without manually calculating each CIDR block.

```hcl
variable "vpc_cidr" {
  default = "10.0.0.0/16"
}

variable "azs" {
  default = ["us-east-1a", "us-east-1b", "us-east-1c"]
}

resource "aws_vpc" "main" {
  cidr_block = var.vpc_cidr
}

resource "aws_subnet" "public" {
  count             = length(var.azs)
  vpc_id            = aws_vpc.main.id
  availability_zone = var.azs[count.index]
  # 10.0.0.0/24, 10.0.1.0/24, 10.0.2.0/24
  cidr_block        = cidrsubnet(var.vpc_cidr, 8, count.index)
}

resource "aws_subnet" "private" {
  count             = length(var.azs)
  vpc_id            = aws_vpc.main.id
  availability_zone = var.azs[count.index]
  # 10.0.10.0/24, 10.0.11.0/24, 10.0.12.0/24
  cidr_block        = cidrsubnet(var.vpc_cidr, 8, count.index + 10)
}

resource "aws_subnet" "database" {
  count             = length(var.azs)
  vpc_id            = aws_vpc.main.id
  availability_zone = var.azs[count.index]
  # 10.0.20.0/24, 10.0.21.0/24, 10.0.22.0/24
  cidr_block        = cidrsubnet(var.vpc_cidr, 8, count.index + 20)
}

output "subnet_layout" {
  value = {
    public   = aws_subnet.public[*].cidr_block
    private  = aws_subnet.private[*].cidr_block
    database = aws_subnet.database[*].cidr_block
  }
}
```

**Console verification:**
```bash
terraform console
> cidrsubnet("10.0.0.0/16", 8, 0)
"10.0.0.0/24"
> cidrsubnet("10.0.0.0/16", 8, 10)
"10.0.10.0/24"
> cidrsubnet("10.0.0.0/16", 8, 20)
"10.0.20.0/24"
```

**Expected outcome:** 9 subnets created with mathematically correct, non-overlapping CIDRs — no manual calculation needed.  
**Exam sub-objective:** Objective 4 — `cidrsubnet()` IP networking function; `count` meta-argument.

</details>

<details>
<summary>Scenario 4: Reading and Rendering an IAM Policy from a JSON Template</summary>

**Situation:**  
A security team maintains IAM policies as JSON templates in a `policies/` directory. Terraform reads and renders these templates with account-specific values using a combination of `templatefile()` and `jsonencode()`.

```json
// policies/s3-access.json.tftpl
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": ["s3:GetObject", "s3:PutObject"],
      "Resource": "arn:aws:s3:::${bucket_name}/*"
    },
    {
      "Effect": "Allow",
      "Action": "s3:ListBucket",
      "Resource": "arn:aws:s3:::${bucket_name}"
    }
  ]
}
```

```hcl
resource "aws_s3_bucket" "app" {
  bucket = "my-app-${var.environment}-assets"
}

resource "aws_iam_policy" "s3_access" {
  name   = "app-s3-access-${var.environment}"
  policy = templatefile("${path.module}/policies/s3-access.json.tftpl", {
    bucket_name = aws_s3_bucket.app.bucket
  })
}

# Alternative approach: build the policy inline with jsonencode()
resource "aws_iam_policy" "s3_access_inline" {
  name = "app-s3-access-inline-${var.environment}"
  policy = jsonencode({
    Version = "2012-10-17"
    Statement = [
      {
        Effect   = "Allow"
        Action   = ["s3:GetObject", "s3:PutObject"]
        Resource = "${aws_s3_bucket.app.arn}/*"
      },
      {
        Effect   = "Allow"
        Action   = "s3:ListBucket"
        Resource = aws_s3_bucket.app.arn
      }
    ]
  })
}
```

**Expected outcome:** The IAM policy document is dynamically populated with the actual bucket name at plan time — no hardcoded ARNs.  
**Exam sub-objective:** Objective 4 — `templatefile()` and `jsonencode()` encoding/filesystem functions.

</details>

<details>
<summary>Scenario 5: Flattening a Multi-VPC Subnet List for Cross-Resource References</summary>

**Situation:**  
A team provisions multiple VPCs, each with several subnets. A downstream resource (a Transit Gateway attachment) needs a flat list of all subnet IDs across all VPCs.

```hcl
variable "vpc_configs" {
  type = map(object({
    cidr        = string
    subnet_count = number
  }))
  default = {
    vpc_a = { cidr = "10.0.0.0/16",  subnet_count = 3 }
    vpc_b = { cidr = "10.1.0.0/16",  subnet_count = 2 }
    vpc_c = { cidr = "10.2.0.0/16",  subnet_count = 3 }
  }
}

# After creating subnets, collect all subnet IDs per VPC into a nested structure:
# [
#   ["subnet-aaa", "subnet-bbb", "subnet-ccc"],  # vpc_a subnets
#   ["subnet-ddd", "subnet-eee"],                 # vpc_b subnets
#   ["subnet-fff", "subnet-ggg", "subnet-hhh"],  # vpc_c subnets
# ]

locals {
  # flatten() produces a single list from the nested structure
  all_subnet_ids = flatten([
    for vpc_name, vpc in var.vpc_configs : [
      # Normally this would reference actual resource attributes:
      # aws_subnet.main["${vpc_name}"].id
      # Shown here conceptually:
      [for i in range(vpc.subnet_count) : "subnet-${vpc_name}-${i}"]
    ]
  ])
  # Result: ["subnet-vpc_a-0", "subnet-vpc_a-1", "subnet-vpc_a-2",
  #          "subnet-vpc_b-0", "subnet-vpc_b-1",
  #          "subnet-vpc_c-0", "subnet-vpc_c-1", "subnet-vpc_c-2"]
}

output "all_subnets" {
  value = local.all_subnet_ids
}
```

**Console verification:**
```bash
terraform console
> flatten([["a","b"], ["c"], ["d","e"]])
["a", "b", "c", "d", "e"]
```

**Expected outcome:** `flatten()` collapses the nested list structure into a single flat list of subnet IDs, which can then be passed to resources requiring a `list(string)`.  
**Exam sub-objective:** Objective 4 — `flatten()` collection function; `for` expressions; `range()` function.

</details>

---
## 14. Quick-Reference Cheat Sheet

```
╔═══════════════════════════════════════════════════════════════════════════════╗
║          TERRAFORM BUILT-IN FUNCTIONS — EXAM CHEAT SHEET                    ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ TESTING: terraform console — REPL for testing expressions without applying  ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ STRING                                                                       ║
║  upper/lower  trimspace  split/join  replace  substr  format  formatlist     ║
║  regex/regexall  startswith  endswith                                        ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ NUMERIC                                                                      ║
║  abs  ceil  floor  max  min  parseint(str, base)  signum                    ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ COLLECTION ⭐                                                                ║
║  length()     — works on lists, maps, sets, strings                         ║
║  contains()   — check membership in a list/set                              ║
║  keys()/values() — decompose a map                                          ║
║  lookup(map, key, default) ⭐ — safe map access with fallback               ║
║  merge(m1,m2,...) ⭐ — combine maps; last key wins on conflict              ║
║  flatten(nested_list) ⭐ — collapse nested lists to single flat list        ║
║  distinct()   — remove duplicates from list                                 ║
║  concat()     — combine multiple lists                                      ║
║  element(list, index) — index with modulo wrap-around                      ║
║  index(list, value) — find position; errors if not found                   ║
║  range()      — generate integer sequence                                   ║
║  zipmap(keys, values) — build map from two parallel lists                  ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ TYPE CONVERSION                                                              ║
║  tostring()  tonumber()  tobool()                                           ║
║  tolist()  toset()  tomap()                                                 ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ ENCODING ⭐                                                                  ║
║  jsonencode(val) ⭐ — HCL → JSON string (inline IAM policies)               ║
║  jsondecode(str)   — JSON string → HCL value                               ║
║  yamlencode/yamldecode — HCL ↔ YAML                                        ║
║  base64encode/base64decode                                                  ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ FILESYSTEM ⭐                                                                ║
║  file(path)          — read file as string                                  ║
║  filebase64(path)    — read file as base64                                  ║
║  templatefile(path, vars) ⭐ — read + render template variables             ║
║  path.module / path.root / path.cwd — standard path references             ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ DATE/TIME                                                                    ║
║  timestamp() — RFC 3339 UTC; changes every plan → use ignore_changes        ║
║  formatdate(format, timestamp)                                              ║
║  timeadd(timestamp, duration)                                               ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ IP NETWORKING ⭐                                                             ║
║  cidrsubnet(prefix, newbits, netnum) ⭐ — calculate subnet CIDR             ║
║    cidrsubnet("10.0.0.0/16", 8, 3) → "10.0.3.0/24"                        ║
║  cidrhost(prefix, hostnum)  — host IP within CIDR                          ║
║  cidrnetmask(prefix)        — subnet mask string                           ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ CONDITIONAL / LOGIC                                                          ║
║  condition ? true_val : false_val  — inline if/else                        ║
║  can(expr)   — bool: true if expr evaluates without error (use in valid.)  ║
║  try(e1,e2,..,fallback) — return first expr that succeeds                  ║
╚═══════════════════════════════════════════════════════════════════════════════╝
```